In [1]:
# notebook : semdetect_e1.ipynb — cellule 0 : imports et configuration
import json
import logging
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score

from src.config import PROJ_ROOT
from src.data.dataset_semdetect import SemDetectDataset
from src.detector.sem_detect.claims_llm import LocalClaimExtractor
from src.detector.sem_detect.feature_store import SemDetectFeatureStore, migrate_legacy_store
from src.detector.sem_detect.references import build_feature_matrix, paper_protocol_references, predict_proba
from src.detector.sem_detect.semdetect.classifier import FEATURE_DISPLAY_NAMES, SemDetectClassifier
from src.evaluation.metrics import get_tpr_at_fpr
from src.utils.vllm_manager import VLLMManager

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)

CACHE_ROOT = Path(os.environ["CACHE_DIR"])
STORE_ROOT = CACHE_ROOT / "sem_detect_store"
REVISION = "18b74b551670bff6b04e57570232a1694997863b"
HF_SOURCE = f"hf:Sem-Detect/ML_Conferences-Peer-Reviews@{REVISION}"
TEXT_SOURCE = f"{HF_SOURCE}:clean_review"   # textes du dataset, sans nettoyage supplémentaire
FEATURES_DEVICE = "cuda:0"                 # Mistral + embeddings ; vLLM est sur le GPU 1

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-04 21:04:25.033 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


In [2]:
# notebook : semdetect_e1.ipynb — cellule 1 : migration du cache E0 (une seule fois, idempotente)
for row in migrate_legacy_store(STORE_ROOT, text_source=TEXT_SOURCE):
    print(row)

store_e0 = SemDetectFeatureStore(root=STORE_ROOT, claims_source=HF_SOURCE, text_source=TEXT_SOURCE,
                                 device=FEATURES_DEVICE)
print(f"E0 : {store_e0.n_embeddings()} embeddings | {store_e0.n_textual()} stats textuelles (attendu : 4410 / 4410)")

{'legacy': '2791ce73dd94', 'kind': 'emb', 'target': '0295060840ff', 'action': 'déjà migré'}
{'legacy': '2791ce73dd94', 'kind': 'txt', 'target': '2af061950803', 'action': 'déjà migré'}


INFO:src.detector.sem_detect.feature_store:Feature store : 4410 embeddings (0295060840ff), 4410 stats textuelles (2af061950803)


E0 : 4410 embeddings | 4410 stats textuelles (attendu : 4410 / 4410)


In [3]:
# notebook : semdetect_e1.ipynb — cellule 2 : test Sem-Detect et références (identiques à E0)
te = SemDetectDataset(config={
    "name": "sem_detect", "repo_id": "Sem-Detect/ML_Conferences-Peer-Reviews", "revision": REVISION,
    "split": "test", "conference": None, "date": None, "generator": None,
    "label_mode": "three_class", "balance": "none", "subset_size": None,
    "val_fraction": None, "val_seed": 0,
    "no_duplicate": True, "keep_meta": True, "with_paper": False, "keep_claims": True,
    "exclude_ids_path": None, "seed": 0,
}).to_pandas()
te_idx = te.set_index("review-id")
refs = paper_protocol_references(te, k=3, seed=0)
clf = SemDetectClassifier(str(PROJ_ROOT / "src/detector/sem_detect/models/lightgbm_best_model.pkl"))
print(f"{len(te)} reviews | {te['class'].value_counts().to_dict()}")

INFO:src.data.dataset_semdetect:📦 Loading Sem-Detect dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_deb985b4e19d6e357800ffa800e5bf14.pt
INFO:src.detector.sem_detect.semdetect.classifier:Loading Sem-Detect model from /home/cyaiche/generated_text_detection_hub/src/detector/sem_detect/models/lightgbm_best_model.pkl
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to br

4410 reviews | {'rewrite': 2440, 'ai': 1364, 'human': 606}


In [4]:
# notebook : semdetect_e1.ipynb — cellule 3 : extraction des claims (GPU 1, ~2 h 15, reprenable)
mgr = VLLMManager(gpu=1, restart_watchdog=True)
mgr.ensure("qwen3-32b-batch")

extractor = LocalClaimExtractor(cache_root=CACHE_ROOT, served_model="qwen3-32b", max_workers=32)
local_claims = extractor.extract(te)
print(f"source : {extractor.source_tag}")
print(local_claims["claims_status"].value_counts().to_dict())

failed = local_claims[local_claims["claims_status"] != "ok"].merge(te[["review-id", "class"]], on="review-id")
if not failed.empty:
    print("échecs par classe :", failed["class"].value_counts().to_dict())

INFO:src.utils.vllm_manager:qwen3-32b déjà servi et opérationnel : réutilisé
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 4410 ok


source : vllm:qwen3-32b:0902bb5d60
{'ok': 4410}


In [5]:
# notebook : semdetect_e1.ipynb — cellule 4 : embeddings des claims locales (GPU 0) ; stats textuelles partagées
te_local = te[["review-id", "content"]].merge(local_claims[["review-id", "claim_extraction"]], on="review-id")
store_e1 = SemDetectFeatureStore(root=STORE_ROOT, claims_source=extractor.source_tag, text_source=TEXT_SOURCE,
                                 device=FEATURES_DEVICE)
assert store_e1.txt_dir == store_e0.txt_dir, "les features textuelles doivent être partagées entre E0 et E1"
store_e1.ensure_embeddings(te_local)
store_e1.ensure_textual(te)  # doit être instantané : cache textuel commun
print(f"E1 : {store_e1.n_embeddings()} embeddings | {store_e1.n_textual()} stats textuelles")

INFO:src.detector.sem_detect.feature_store:Feature store : 4410 embeddings (fbbbe1608d0e), 4410 stats textuelles (2af061950803)


E1 : 4410 embeddings | 4410 stats textuelles


In [6]:
# notebook : semdetect_e1.ipynb — cellule 5 : E0 vs E1, mêmes cibles, mêmes références, même classifieur
CLASS_TO_IDX = {"ai": 0, "rewrite": 1, "human": 2}
NAMES = ["AI", "LLM-Refined", "Human"]


def run(store: SemDetectFeatureStore):
    X, kept, skipped = build_feature_matrix(store, te["review-id"].tolist(), refs, clf.similarity_threshold)
    P = pd.DataFrame(predict_proba(clf, X), index=kept, columns=["p_ai", "p_rewrite", "p_human"])
    return P, pd.DataFrame(X, index=kept, columns=FEATURE_DISPLAY_NAMES), skipped


def metrics(P: pd.DataFrame) -> dict:
    cls = te_idx.loc[P.index, "class"]
    y, yp = cls.map(CLASS_TO_IDX).to_numpy(), P.to_numpy().argmax(axis=1)
    prec, rec, _, _ = precision_recall_fscore_support(y, yp, labels=[0, 1, 2], zero_division=0)
    p_ai, is_ai = P["p_ai"].to_numpy(), (cls == "ai").to_numpy()
    m = cls.isin(["human", "ai"]).to_numpy()
    return {
        "n classées": len(y), "macro-F1": f1_score(y, yp, average="macro"),
        "P_ai": prec[0], "R_ai": rec[0], "P_rewrite": prec[1], "R_rewrite": rec[1], "P_human": prec[2], "R_human": rec[2],
        "AUC humain/IA": roc_auc_score(is_ai[m], p_ai[m]), "TPR@1% humain/IA": get_tpr_at_fpr(is_ai[m], p_ai[m], 0.01),
        "AUC (h+r)/IA": roc_auc_score(is_ai, p_ai),
    }


results = {"E0 Gemini": run(store_e0), "E1 qwen3-32b": run(store_e1)}

print("=== Métriques (chaque configuration sur ses propres cibles classées) ===")
print(pd.DataFrame({name: metrics(P) for name, (P, _, _) in results.items()}).round(4), "\n")

print("=== Cibles non classables ===")
for name, (_, _, skipped) in results.items():
    s = pd.Series(skipped, dtype=object)
    by = s.to_frame("raison").join(te_idx["class"]).value_counts() if len(s) else "aucune"
    print(f"{name} : {len(s)}\n{by}\n")

print("=== Confusion (% par ligne vraie) ===")
for name, (P, _, _) in results.items():
    cls = te_idx.loc[P.index, "class"]
    cm = confusion_matrix(cls.map(CLASS_TO_IDX), P.to_numpy().argmax(axis=1), labels=[0, 1, 2], normalize="true")
    print(f"{name}\n{pd.DataFrame(np.round(cm * 100, 2), index=NAMES, columns=NAMES)}\n")

# comparaison appariée sur les cibles classées dans les deux configurations
(P0, X0, _), (P1, X1, _) = results["E0 Gemini"], results["E1 qwen3-32b"]
common = P0.index.intersection(P1.index)
y = te_idx.loc[common, "class"].map(CLASS_TO_IDX).to_numpy()
yp0, yp1 = P0.loc[common].to_numpy().argmax(axis=1), P1.loc[common].to_numpy().argmax(axis=1)
rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    diffs.append(f1_score(y[i], yp1[i], average="macro") - f1_score(y[i], yp0[i], average="macro"))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"=== Apparié ({len(common)} cibles communes) ===")
print(f"Δ macro-F1 (E1 − E0) = {f1_score(y, yp1, average='macro') - f1_score(y, yp0, average='macro'):+.4f} "
      f"[IC95 bootstrap : {lo:+.4f}, {hi:+.4f}]")
print(f"accord des prédictions E0/E1 : {(yp0 == yp1).mean():.1%}\n")

print("=== Moyenne des 9 features (référence : moyenne du train du classifieur) ===")
print(pd.DataFrame({
    "train (scaler)": clf.scaler.mean_,
    "E0 Gemini": X0.loc[common].mean().to_numpy(),
    "E1 qwen3-32b": X1.loc[common].mean().to_numpy(),
}, index=FEATURE_DISPLAY_NAMES).round(4))

=== Métriques (chaque configuration sur ses propres cibles classées) ===
                  E0 Gemini  E1 qwen3-32b
n classées        4373.0000     4362.0000
macro-F1             0.8418        0.8259
P_ai                 0.9381        0.9590
R_ai                 0.9113        0.8434
P_rewrite            0.8725        0.8388
R_rewrite            0.9232        0.9476
P_human              0.7751        0.8040
R_human              0.6444        0.6047
AUC humain/IA        0.9980        0.9963
TPR@1% humain/IA     0.9282        0.9272
AUC (h+r)/IA         0.9912        0.9902 

=== Cibles non classables ===
E0 Gemini : 37
raison                        class  
cible < 4 claims sémantiques  rewrite    30
                              human       7
Name: count, dtype: int64

E1 qwen3-32b : 48
raison                          class  
cible < 4 claims sémantiques    rewrite    32
                                human       8
références insuffisantes (2/3)  rewrite     3
                           

In [7]:
# notebook : semdetect_e1.ipynb — cellule 6 : diagnostic et relance ciblée des échecs (GPU 1)
extractor = LocalClaimExtractor(cache_root=CACHE_ROOT, served_model="qwen3-32b", max_workers=32)
recs = extractor.records()
recs = recs[recs.index.isin(te["review-id"])]
failed_ids = list(recs.index[recs["status"] != "ok"])
print(f"{len(failed_ids)} échec(s) :")
for rid in failed_ids:
    print(f"  {rid} ({te_idx.loc[rid, 'class']}, {len(te_idx.loc[rid, 'content'])} car.) : {recs.loc[rid, 'error']}")

mgr = VLLMManager(gpu=1, restart_watchdog=True)
mgr.ensure("qwen3-32b-batch")

# 1) sortie simplement longue : limite relevée (MAX_MODEL_LEN=16384 couvre prompt + 8192)
local_claims = extractor.retry_failed(te, max_tokens=8192)
print("après relance 1 :", local_claims["claims_status"].value_counts().to_dict())

# 2) génération en boucle : légère pénalité de répétition
local_claims = extractor.retry_failed(te, max_tokens=8192, extra_sampling={"repetition_penalty": 1.05})
print("après relance 2 :", local_claims["claims_status"].value_counts().to_dict())

recs_after = extractor.records().loc[failed_ids]
print(recs_after[["status", "attempts", "overrides"]].to_string())

INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 4410 ok


0 échec(s) :


INFO:src.utils.vllm_manager:qwen3-32b déjà servi et opérationnel : réutilisé
INFO:src.detector.sem_detect.claims_llm:relance de 0 échec(s) avec max_tokens=8192, extra=None
INFO:src.detector.sem_detect.claims_llm:relance de 0 échec(s) avec max_tokens=8192, extra={'repetition_penalty': 1.05}


après relance 1 : {'ok': 4410}
après relance 2 : {'ok': 4410}
Empty DataFrame
Columns: [status, attempts, overrides]
Index: []


In [8]:
# notebook : semdetect_e1.ipynb — cellule 7 : recalcul des embeddings des reviews relancées (GPU 0)
te_local = te[["review-id", "content"]].merge(local_claims[["review-id", "claim_extraction"]], on="review-id")
store_e1.ensure_embeddings(te_local, force_ids=failed_ids)
still_empty = [rid for rid in failed_ids if store_e1.semantic(rid, 1) is None]
print(f"{len(failed_ids) - len(still_empty)} / {len(failed_ids)} reviews relancées ont désormais des claims sémantiques")

0 / 0 reviews relancées ont désormais des claims sémantiques


In [9]:
# notebook : semdetect_e1.ipynb — cellule 7b : rechargement de references.py
import importlib
import inspect

import src.detector.sem_detect.references as references_mod

references_mod = importlib.reload(references_mod)
from src.detector.sem_detect.references import build_feature_matrix, paper_protocol_references, predict_proba

print("fichier :", references_mod.__file__)
print("signature :", inspect.signature(paper_protocol_references))
assert "is_valid" in inspect.signature(paper_protocol_references).parameters, \
    "references.py n'est pas à jour à cet emplacement : vérifier que le fichier a bien été enregistré"

fichier : /home/cyaiche/generated_text_detection_hub/src/detector/sem_detect/references.py
signature : (df: 'pd.DataFrame', k: 'int' = 3, seed: 'int' = 0, is_valid: 'Optional[Callable[[str], bool]]' = None) -> 'Dict[str, List[str]]'


In [10]:
# notebook : semdetect_e1.ipynb — cellule 8 : réévaluation avec références valides
def valid_in(store: SemDetectFeatureStore):
    return lambda rid: store.semantic(rid, 1) is not None


refs_e0 = paper_protocol_references(te, k=3, seed=0, is_valid=valid_in(store_e0))
refs_e1 = paper_protocol_references(te, k=3, seed=0, is_valid=valid_in(store_e1))

# non-régression : avec les claims Gemini, aucune candidate n'est invalide -> références inchangées
n_diff_e0 = sum(refs_e0[r] != refs[r] for r in refs)
print(f"E0 : {n_diff_e0} cibles dont les références changent (attendu : 0)")
print(f"E1 : {sum(refs_e1[r] != refs[r] for r in refs)} cibles dont les références changent\n")


def run_with(store: SemDetectFeatureStore, refs_map):
    X, kept, skipped = build_feature_matrix(store, te["review-id"].tolist(), refs_map, clf.similarity_threshold)
    P = pd.DataFrame(predict_proba(clf, X), index=kept, columns=["p_ai", "p_rewrite", "p_human"])
    return P, pd.DataFrame(X, index=kept, columns=FEATURE_DISPLAY_NAMES), skipped


results_a = {"E0 Gemini": run_with(store_e0, refs_e0), "E1 qwen3-32b (A)": run_with(store_e1, refs_e1)}

print("=== Métriques ===")
print(pd.DataFrame({name: metrics(P) for name, (P, _, _) in results_a.items()}).round(4), "\n")

print("=== Cibles non classables ===")
for name, (_, _, skipped) in results_a.items():
    s = pd.Series(skipped, dtype=object)
    by = s.to_frame("raison").join(te_idx["class"]).value_counts() if len(s) else "aucune"
    print(f"{name} : {len(s)}\n{by}\n")

(P0, _, _), (P1, _, _) = results_a["E0 Gemini"], results_a["E1 qwen3-32b (A)"]
common = P0.index.intersection(P1.index)
y = te_idx.loc[common, "class"].map(CLASS_TO_IDX).to_numpy()
yp0, yp1 = P0.loc[common].to_numpy().argmax(axis=1), P1.loc[common].to_numpy().argmax(axis=1)
rng = np.random.default_rng(0)
diffs = []
for _ in range(1000):
    i = rng.integers(0, len(y), len(y))
    diffs.append(f1_score(y[i], yp1[i], average="macro") - f1_score(y[i], yp0[i], average="macro"))
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"=== Apparié ({len(common)} cibles communes) ===")
print(f"Δ macro-F1 (E1 − E0) = {f1_score(y, yp1, average='macro') - f1_score(y, yp0, average='macro'):+.4f} "
      f"[IC95 bootstrap : {lo:+.4f}, {hi:+.4f}]")

E0 : 0 cibles dont les références changent (attendu : 0)
E1 : 7 cibles dont les références changent

=== Métriques ===
                  E0 Gemini  E1 qwen3-32b (A)
n classées        4373.0000         4366.0000
macro-F1             0.8418            0.8258
P_ai                 0.9381            0.9590
R_ai                 0.9113            0.8434
P_rewrite            0.8725            0.8389
R_rewrite            0.9232            0.9473
P_human              0.7751            0.8027
R_human              0.6444            0.6054
AUC humain/IA        0.9980            0.9963
TPR@1% humain/IA     0.9282            0.9272
AUC (h+r)/IA         0.9912            0.9902 

=== Cibles non classables ===
E0 Gemini : 37
raison                        class  
cible < 4 claims sémantiques  rewrite    30
                              human       7
Name: count, dtype: int64

E1 qwen3-32b (A) : 44
raison                          class  
cible < 4 claims sémantiques    rewrite    32
                     